<a href="https://colab.research.google.com/github/anthony-yt/deepwater-image-segmentation-classical-quantum/blob/main/notebooks/1_optical_preprocessing_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q albumentations

In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/Under_Water_Dataset"
FRAMES_RAW_DIR = os.path.join(DRIVE_ROOT, "Dataset_Processed", "Frames_Crudos")
DATASET_FINAL_DIR = os.path.join(DRIVE_ROOT, "Dataset_Processed", "Dataset_Final_512")

os.makedirs(FRAMES_RAW_DIR, exist_ok=True)
os.makedirs(DATASET_FINAL_DIR, exist_ok=True)

Mounted at /content/drive


In [3]:
def extract_video_frames(input_dir: str, output_dir: str, interval_sec: int = 2) -> int:
    """Extract frames from underwater videos at a fixed temporal sampling rate."""
    if len(os.listdir(output_dir)) > 0:
        return len(os.listdir(output_dir))

    valid_extensions = ('.mp4', '.avi', '.mov')
    video_files = [f for f in os.listdir(input_dir) if f.lower().endswith(valid_extensions)]
    total_extracted = 0

    for video_name in video_files:
        video_path = os.path.join(input_dir, video_name)
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            continue

        frame_step = int(cap.get(cv2.CAP_PROP_FPS) * interval_sec)
        frame_count, saved_count = 0, 0
        base_name = os.path.splitext(video_name)[0]

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            if frame_count % frame_step == 0:
                out_path = os.path.join(output_dir, f"{base_name}_frame_{saved_count:04d}.jpg")
                cv2.imwrite(out_path, frame)
                saved_count += 1
                total_extracted += 1

            frame_count += 1

        cap.release()

    return total_extracted

total_frames = extract_video_frames(DRIVE_ROOT, FRAMES_RAW_DIR, interval_sec=2)
print(f"Extracted frames available: {total_frames}")

Extracted frames available: 105


In [4]:
def apply_clahe(img_bgr: np.ndarray, clip_limit: float = 3.0, grid_size: tuple = (8, 8)) -> np.ndarray:
    """Apply CLAHE solely on the luminance channel (L) in CIELAB space."""
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l_clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=grid_size).apply(l)
    return cv2.cvtColor(cv2.merge((l_clahe, a, b)), cv2.COLOR_LAB2BGR)

def apply_msrcr(img_bgr: np.ndarray, sigmas: list = [15, 80, 250]) -> np.ndarray:
    """Apply Multi-Scale Retinex with Color Restoration using normalized dynamic range clipping."""
    img_float = np.float64(img_bgr) + 1.0
    msr = np.zeros_like(img_float)

    for sigma in sigmas:
        blur = cv2.GaussianBlur(img_float, (0, 0), sigma)
        msr += np.log10(img_float) - np.log10(blur)
    msr = msr / len(sigmas)

    img_sum = np.sum(img_float, axis=2, keepdims=True)
    color_restoration = np.log10(125.0 * img_float) - np.log10(img_sum)
    msrcr_raw = 46.0 * (msr * color_restoration) + 1.0

    output = np.zeros_like(msrcr_raw)
    for c in range(3):
        channel = msrcr_raw[:, :, c]
        p_low, p_high = np.percentile(channel, (1, 99))
        channel_clipped = np.clip(channel, p_low, p_high)
        output[:, :, c] = cv2.normalize(channel_clipped, None, 0, 255, cv2.NORM_MINMAX)

    return output.astype(np.uint8)

def apply_udcp(img_bgr: np.ndarray, kernel_size: int = 15) -> np.ndarray:
    """Apply Underwater Dark Channel Prior heuristic to estimate transmission."""
    dark_channel = cv2.min(img_bgr[:, :, 0], img_bgr[:, :, 1])
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))
    dark_prior = cv2.erode(dark_channel, kernel)

    img_norm = img_bgr.astype(np.float32) / 255.0
    transmission_est = (dark_prior[..., None] / 255.0) * 0.5
    restored = cv2.normalize(img_norm - transmission_est, None, 0, 255, cv2.NORM_MINMAX)

    return restored.astype(np.uint8)

In [5]:
output_dirs = {
    "clahe": os.path.join(DRIVE_ROOT, "Dataset_Processed", "Frames_CLAHE"),
    "msrcr": os.path.join(DRIVE_ROOT, "Dataset_Processed", "Frames_MSRCR"),
    "udcp": os.path.join(DRIVE_ROOT, "Dataset_Processed", "Frames_UDCP")
}

for out_path in output_dirs.values():
    os.makedirs(out_path, exist_ok=True)

extracted_files = sorted([f for f in os.listdir(FRAMES_RAW_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
processed_count = len(os.listdir(output_dirs["clahe"]))

if processed_count < len(extracted_files):
    for file_name in extracted_files:
        img_path = os.path.join(FRAMES_RAW_DIR, file_name)
        raw_frame = cv2.imread(img_path)

        if raw_frame is not None:
            cv2.imwrite(os.path.join(output_dirs["clahe"], file_name), apply_clahe(raw_frame))
            cv2.imwrite(os.path.join(output_dirs["msrcr"], file_name), apply_msrcr(raw_frame))
            cv2.imwrite(os.path.join(output_dirs["udcp"], file_name), apply_udcp(raw_frame))
    print(f"Batch processing completed for {len(extracted_files)} frames.")
else:
    print(f"Batch processing bypassed. {processed_count} frames already filtered.")

Batch processing bypassed. 105 frames already filtered.


In [6]:
def audit_ground_truth(directories: list) -> tuple[pd.DataFrame, float]:
    """Audit the presence of manual ground truth segmentation masks across multiple directories."""
    valid_extensions = ('.jpg', '.jpeg', '.png')
    records = []
    total_valid = 0
    total_images = 0

    for dataset_dir in directories:
        if not os.path.exists(dataset_dir):
            continue

        all_files = os.listdir(dataset_dir)
        base_images = [f for f in all_files if f.lower().endswith(valid_extensions) and "mask" not in f.lower()]

        for img_name in base_images:
            total_images += 1
            base_name, ext = os.path.splitext(img_name)
            mask_candidates = [f"{base_name}_mask{ext}", f"{base_name}_mask.png"]

            status = "Missing"
            for candidate in mask_candidates:
                if candidate in all_files:
                    mask_data = cv2.imread(os.path.join(dataset_dir, candidate), cv2.IMREAD_GRAYSCALE)
                    if mask_data is not None:
                        unique_pixels = np.unique(mask_data)
                        status = "Valid (Binary)" if len(unique_pixels) <= 2 else "Invalid (Non-Binary)"
                        total_valid += 1
                    else:
                        status = "Corrupted"
                    break

            records.append({"Directory": os.path.basename(dataset_dir), "Image": img_name, "Mask Status": status})

    df = pd.DataFrame(records)
    coverage = (total_valid / total_images) * 100 if total_images > 0 else 0.0
    return df, coverage

dirs_to_audit = [DRIVE_ROOT, FRAMES_RAW_DIR]
df_audit, mask_coverage = audit_ground_truth(dirs_to_audit)

print(f"Comprehensive ground truth coverage: {mask_coverage:.2f}%\n")

Comprehensive ground truth coverage: 0.00%



In [10]:
def evaluate_image_quality(image_bgr: np.ndarray) -> tuple[float, float]:
    """Calculate spatial variance and entropy as proxies for underwater quality."""
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    variance = np.var(gray)

    hist = cv2.calcHist([gray], [0], None, [256], [0, 256]).ravel()
    hist = hist[hist > 0] / hist.sum()
    entropy = -1.0 * np.sum(hist * np.log2(hist))

    return variance, entropy

extracted_files = sorted([f for f in os.listdir(FRAMES_RAW_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

if len(extracted_files) > 0:
    sample_frame = extracted_files[0]
    raw_path = os.path.join(FRAMES_RAW_DIR, sample_frame)
    raw_img = cv2.imread(raw_path)

    processed_imgs = {
        "Raw Frame": raw_img,
        "CLAHE": apply_clahe(raw_img),
        "MSRCR": apply_msrcr(raw_img),
        "UDCP": apply_udcp(raw_img)
    }

    print(f"Evaluating sample: {sample_frame}\n")
    print(f"{'Filter':<15} | {'Variance (Sharpness)':<20} | {'Entropy (Detail)':<20}")
    print("-" * 60)
    for name, img in processed_imgs.items():
        var, ent = evaluate_image_quality(img)
        print(f"{name:<15} | {var:<20.2f} | {ent:<20.2f}")
else:
    print("No raw frames found to evaluate.")

Evaluating sample: DeepWater-Video1_frame_0000.jpg

Filter          | Variance (Sharpness) | Entropy (Detail)    
------------------------------------------------------------
Raw Frame       | 821.66               | 6.22                
CLAHE           | 1002.99              | 6.81                
MSRCR           | 3277.95              | 7.49                
UDCP            | 1214.13              | 6.62                
